# How Much Is There? Turning a Peak Into a Number
### 3.5 — Peak Integration and Quantifying Area

In the last lesson we learned to *find* peaks. This one asks the question every
quantitative measurement eventually comes down to: **how much?** A detected band
tells you *something is there*; integration turns that band into a **number** —
its area — that you can put on a calibration curve and report.

It is tempting to think this lesson is "how to call `numpy.trapezoid()`." It
isn't. That function is one line, and on its own it will happily give you a
confidently wrong answer. The number it returns depends entirely on decisions
*you* make and the simulator quietly makes for us in real life:

- **area vs. height** — why area is usually the steadier measure of "how much",
  and when height misleads;
- **where you draw the integration limits** — too narrow and you truncate the
  band; too wide and you scoop up neighbours and background;
- **the baseline** — integration adds up *everything* in the window, the
  background included, so an uncorrected baseline biases every area;
- **overlapping bands** — when two peaks share one window, area is conserved but
  not separable;
- **uncertainty** — how much the answer wobbles when you nudge the limits, which
  is a number worth reporting;
- and finally **why we do any of this**: area tracks concentration, which is the
  whole point — quantitation.

Because every spectrum here is built with `uvvis.simulate()`, we always **know
the true area** (a Gaussian band's area is exact: `amplitude × FWHM × √(π/4ln2)`)
and the **true concentration**. So we can *grade* each integration against the
right answer instead of trusting that the shaded region looks about right.

> **Where this sits:** this follows **3.3 (AsLS Baseline Correction)** and
> **3.4 (Peak Detection)**. Detection locates the band; baseline correction
> cleans the background *under* it; integration is where those two pay off. We
> reuse both here.

**What we'll cover**

1. Setup
2. Area vs. height — two ways to say "how much"
3. Numerical integration from scratch — the trapezoid
4. Where you start and stop: integration boundaries
5. The baseline problem — integrating the background by accident
6. Correcting the baseline under a peak
7. How much do the boundaries matter? (uncertainty)
8. Overlapping peaks — one window, two bands
9. Why this matters: area, concentration, and calibration
10. A reusable `integrate_peak()` helper

## 1. Setup

In [ ]:
# Standard scientific Python stack.
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Integration lives in numpy/scipy. We mostly use:
#   np.trapezoid  -- the trapezoidal rule (the practical workhorse for spectra)
# In older NumPy this function is spelled np.trapz; we alias it so the notebook
# runs either way.
try:
    trapezoid = np.trapezoid          # NumPy >= 2.0
except AttributeError:
    trapezoid = np.trapz              # older NumPy

# Sparse tools for the AsLS baseline correction we reuse from lesson 3.3.
from scipy import sparse
from scipy.sparse.linalg import spsolve

# Our UV-Vis simulator. It carries its own ground truth, and we also reach into
# the building blocks so we can reconstruct the *exact* clean band and baseline
# behind each spectrum -- that is what lets us grade an integration honestly.
from simulated_data import uvvis
from simulated_data.core.peaks import gaussian, add_peaks
from simulated_data.core.baselines import sloping_baseline

# A folder for saved figures (regenerable scratch; git-ignored).
EXPORTS = Path("exports")
EXPORTS.mkdir(exist_ok=True)

# One consistent, readable plot style for the whole notebook.
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

# A small, consistent colour vocabulary so every figure reads the same way.
C_SIG   = "#1a73e8"  # the measured signal (blue)
C_TRUE  = "#188038"  # ground truth / correct result (green)
C_FALSE = "#ea4335"  # error / bias / problem (red)
C_AUX   = "#f9ab00"  # helper lines, baselines, limits (amber)
C_FILL  = "#aecbfa"  # shaded integration area (light blue)

# A Gaussian band's area is exact and known. With FWHM as the width, the area
# under amplitude * exp(-(x-c)^2 / 2 sigma^2) works out to:
#
#     area = amplitude * FWHM * sqrt(pi / (4 * ln 2))
#
# We will lean on this constant all notebook to compute the *true* area of any
# simulated band -- our answer key.
AREA_K = np.sqrt(np.pi / (4.0 * np.log(2.0)))   # ~= 1.06447

def true_gaussian_area(amplitude, fwhm):
    '''Exact analytic area of one Gaussian band (our ground truth).'''
    return amplitude * fwhm * AREA_K

print(f"Ready. Gaussian area constant = amplitude * FWHM * {AREA_K:.5f}")

## 2. Area vs. height — two ways to say "how much"

There are two obvious numbers you can read off a peak:

- **Height** — the value at the top. One point. Fast, but it throws away the
  entire shape of the band.
- **Area** — the integral, the amount of "stuff" under the curve. It uses every
  point in the band.

For an ideal, fixed-shape band at low concentration both are proportional to
concentration, so either works. The reason analysts reach for **area** is that
it is far more robust when the band's *shape* changes — broadening with
temperature, partial saturation, a shifting baseline. Let's make that concrete.

In [ ]:
# A single, clean (noise-free) band, comfortably inside the 400-800 nm window so
# no tail is clipped. peaks = [(center_nm, FWHM_nm, amplitude_AU)].
band = (550.0, 50.0, 1.0)
ds = uvvis.simulate(peaks=[band], noise=0.0, baseline=None, seed=0)
x = ds.x
y = ds.X[0]                       # X is always 2D (n_samples, n_points); take row 0

height = y.max()                              # the "height" measurement
area   = trapezoid(y, x)                       # the "area" measurement (numeric)
area_true = true_gaussian_area(band[2], band[1])   # the exact analytic area

print(f"height            = {height:.3f} AU            (this is just amplitude)")
print(f"area  (numeric)   = {area:.3f} AU*nm")
print(f"area  (true/exact)= {area_true:.3f} AU*nm")
print(f"agreement         = {100*area/area_true:.2f}% of the exact area")

In [ ]:
# The key demonstration: two bands with the SAME AREA but different HEIGHTS.
# Band A is narrow and tall; band B is broad and short. Choose amplitudes so the
# analytic areas are identical.
target_area = 40.0
A_narrow = (520.0, 30.0, target_area / (30.0 * AREA_K))   # tall, skinny
A_broad  = (620.0, 80.0, target_area / (80.0 * AREA_K))   # short, fat

dsA = uvvis.simulate(peaks=[A_narrow], noise=0.0, baseline=None, seed=1)
dsB = uvvis.simulate(peaks=[A_broad],  noise=0.0, baseline=None, seed=1)
yA, yB = dsA.X[0], dsB.X[0]

fig, ax = plt.subplots()
ax.plot(x, yA, color=C_SIG,  label=f"narrow band  (height {yA.max():.2f}, area {trapezoid(yA, x):.1f})")
ax.plot(x, yB, color=C_TRUE, label=f"broad band   (height {yB.max():.2f}, area {trapezoid(yB, x):.1f})")
ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})"); ax.set_ylabel(ds.y_label)
ax.set_title("Same amount of analyte, very different heights")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "02_area_vs_height.png", dpi=130)
plt.show()

print("Heights differ by ~2.7x; areas are equal. If these were the SAME compound")
print("measured under two conditions, height would call them different amounts.")
print("Area would (correctly) call them the same.")

That is the whole argument for area in one picture. Height answers "how tall is
the tallest point"; area answers "how much is there". When you can trust the band
shape to never change, height is a fine, cheap proxy. The moment shape can drift —
and on real instruments it always can — **area is the more honest measurement.**

## 3. Numerical integration from scratch — the trapezoid

We never have a formula for a real spectrum — just a list of `(x, y)` points. So
"area" means **numerical integration**: approximate the curve between samples by
a simple shape and add up the pieces.

The **trapezoidal rule** connects each pair of neighbouring points with a
straight line and sums the little trapezoids. For points spaced `dx` apart, each
trapezoid has area `(y[i] + y[i+1]) / 2 * dx`. Let's build it by hand once, so
`np.trapezoid` is never a black box again.

In [ ]:
def trapz_by_hand(y, x):
    '''Trapezoidal integration, written out so the rule is visible.'''
    dx = np.diff(x)                     # spacing between neighbouring x points
    avg_height = (y[:-1] + y[1:]) / 2.0  # mean of each adjacent pair of y values
    return np.sum(avg_height * dx)       # sum of trapezoid areas

mine  = trapz_by_hand(y, x)
numpy_ = trapezoid(y, x)
print(f"hand-written trapezoid = {mine:.4f}")
print(f"np.trapezoid           = {numpy_:.4f}")
print(f"difference             = {abs(mine - numpy_):.2e}  (identical: same rule)")

In [ ]:
# Sampling density matters. Integrate the SAME band on progressively coarser
# grids by subsampling, and watch the trapezoid approximation converge.
print(f"{'points':>8} {'spacing(nm)':>12} {'area':>10} {'% of true':>10}")
for step in (40, 20, 8, 4, 2, 1):
    xs, ys = x[::step], y[::step]
    a = trapezoid(ys, xs)
    print(f"{xs.size:>8} {np.diff(xs).mean():>12.2f} {a:>10.3f} {100*a/area_true:>9.2f}%")

With only a handful of points across the band the trapezoid *underestimates* the
rounded top and the answer is visibly low; by the time we have the band well
sampled (a point every nm or two) it is within a fraction of a percent of the
exact area. The lesson: the integration **rule** is not where error usually comes
from — adequate sampling and, as we'll see next, your **limits** and your
**baseline** are. `np.trapezoid` is the right tool; the judgment is yours.

## 4. Where you start and stop: integration boundaries

A peak has tails that technically run forever, but you integrate over a finite
window `[lo, hi]`. Choosing that window is the first real decision. Too narrow
and you chop off the tails (area too low); too wide and you start summing
neighbouring bands and background (area too high — coming up next).

Here is the clean band again. A Gaussian's width is often quoted in `sigma`;
recall `FWHM = 2.355 * sigma`. Let's integrate over a few symmetric windows and
see what fraction of the *true* area each captures.

In [ ]:
center, fwhm = 550.0, 50.0
sigma = fwhm / (2.0 * np.sqrt(2.0 * np.log(2.0)))   # convert FWHM -> sigma

def integrate_window(x, y, lo, hi):
    '''Trapezoid area of y over the wavelength window [lo, hi].'''
    m = (x >= lo) & (x <= hi)        # boolean mask selecting the window
    return trapezoid(y[m], x[m])

rows = []
for k in (1, 2, 3):
    lo, hi = center - k*sigma, center + k*sigma
    a = integrate_window(x, y, lo, hi)
    rows.append((f"+/-{k} sigma", f"{lo:.0f}-{hi:.0f}", a, 100*a/area_true))
# the full axis, for comparison
a_full = integrate_window(x, y, x.min(), x.max())
rows.append(("full axis", f"{x.min():.0f}-{x.max():.0f}", a_full, 100*a_full/area_true))

print(f"{'window':>10} {'range(nm)':>12} {'area':>9} {'% of true area':>16}")
for name, rng, a, pct in rows:
    print(f"{name:>10} {rng:>12} {a:>9.2f} {pct:>15.1f}%")

In [ ]:
# Visualize what each window keeps. The familiar 68 / 95 / 99.7 of a Gaussian
# shows up directly as captured area.
fig, ax = plt.subplots()
ax.plot(x, y, color=C_SIG, lw=1.5)
for k, c in zip((1, 2, 3), (C_FALSE, C_AUX, C_TRUE)):
    lo, hi = center - k*sigma, center + k*sigma
    m = (x >= lo) & (x <= hi)
    ax.fill_between(x[m], y[m], alpha=0.18, color=c, label=f"+/-{k} sigma")
ax.set_xlabel(f"{ds.x_label} ({ds.x_unit})"); ax.set_ylabel(ds.y_label)
ax.set_title("Integration limits decide how much of the band you keep")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "04_boundaries.png", dpi=130)
plt.show()

A `+/-2 sigma` window captures ~95 % of the area; you reach ~99.7 % only by
~`+/-3 sigma`. For an isolated band you'll be baseline-correcting (next section),
the right move is to run the window out to where **the band has returned to the
baseline** — roughly `+/-1.5 * FWHM` (≈ `+/-3.5 sigma`). That isn't just to catch
the last of the area: it puts the window *endpoints* on genuine background rather
than on the peak's tail, which is exactly what the baseline-subtraction methods
need to work. We'll use a `+/-1.5 * FWHM` window from here on.

The point is not the exact number — it's that **the limits are a choice that
changes the answer**, so they must be deliberate and reported. And so far we've
been cheating: the baseline was a flat zero. Real backgrounds are not, and that
is where most integration error actually lives.

## 5. The baseline problem — integrating the background by accident

Integration sums **everything** in the window, and a real spectrum sits on a
background — a sloping drift from lamp ageing or a dirty cuvette. Whatever
baseline runs under your peak gets added straight into the area. Worse, the wider
your window, the more background you scoop up, so the bias is not even constant.

Let's add a sloping baseline and reconstruct the exact pieces (peak-only and
baseline-only) so we can see precisely where the error comes from.

In [ ]:
# Same band, now riding on a sloping baseline, with a little measurement noise.
base_spec = {"type": "sloping", "slope": 0.0020, "offset": 0.05}
dsb = uvvis.simulate(peaks=[band], noise=0.004, baseline=base_spec, seed=5)
xb = dsb.x
yb = dsb.X[0]                       # what the instrument hands you: peak + baseline + noise

# Because this is a simulation, we can rebuild the TRUTH behind that spectrum:
peak_only     = add_peaks(xb, [band])              # the analyte band alone
baseline_only = sloping_baseline(xb, base_spec["slope"], base_spec["offset"])

fig, ax = plt.subplots()
ax.plot(xb, yb, color=C_SIG, lw=1.2, label="measured (peak + baseline + noise)")
ax.plot(xb, baseline_only, color=C_AUX, lw=2, ls="--", label="true baseline (hidden in real life)")
ax.plot(xb, peak_only, color=C_TRUE, lw=1.2, alpha=0.8, label="true peak only")
ax.set_xlabel(f"{dsb.x_label} ({dsb.x_unit})"); ax.set_ylabel(dsb.y_label)
ax.set_title("The measured signal is the peak PLUS a background you didn't ask for")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "05_baseline_signal.png", dpi=130)
plt.show()

In [ ]:
# Integrate the RAW signal and compare to the truth. We use a baseline-to-
# baseline window (+/-1.5 * FWHM), so the endpoints sit on real background -- this
# is the window we'll keep for every baseline-corrected measurement below.
lo, hi = center - 1.5*fwhm, center + 1.5*fwhm
print(f"integration window = {lo:.0f}-{hi:.0f} nm  (+/-1.5*FWHM, ~+/-3.5 sigma)")
m = (xb >= lo) & (xb <= hi)

raw_area      = trapezoid(yb[m], xb[m])            # what naive integration reports
true_peak_win = trapezoid(peak_only[m], xb[m])     # the area we actually want
baseline_win  = trapezoid(baseline_only[m], xb[m]) # the background's contribution

print(f"raw integrated area      = {raw_area:.2f} AU*nm   (peak + background)")
print(f"true peak area in window = {true_peak_win:.2f} AU*nm")
print(f"baseline area in window  = {baseline_win:.2f} AU*nm")
print(f"  check: peak + baseline = {true_peak_win + baseline_win:.2f}  (== raw, up to noise)")
print(f"BIAS from the baseline   = +{100*(raw_area-true_peak_win)/true_peak_win:.1f}% too high")

In [ ]:
# And the bias GROWS with window width: a wider window adds more background area
# while the peak's own contribution has already plateaued.
print(f"{'window':>10} {'raw area':>10} {'true peak':>10} {'bias %':>9}")
for k in (1, 2, 3, 4):
    lo_k, hi_k = center - k*sigma, center + k*sigma
    mk = (xb >= lo_k) & (xb <= hi_k)
    raw_k  = trapezoid(yb[mk], xb[mk])
    peak_k = trapezoid(peak_only[mk], xb[mk])
    print(f"{'+/-'+str(k)+' sig':>10} {raw_k:>10.2f} {peak_k:>10.2f} {100*(raw_k-peak_k)/peak_k:>8.1f}%")

Two things to take away. First, the bias is **positive and large** — here a
modest sloping background nearly *doubles* the reported area. Second, it gets
**worse as the window widens**, which means you cannot fix it by "just integrating
a bit more generously." The only real fix is to remove the baseline *before* you
integrate.

## 6. Correcting the baseline under a peak

Two practical fixes, from quickest to most general:

1. **Local linear ("baseline drop")** — draw a straight line between the signal
   at the two window endpoints and subtract it. This is the classic
   chromatography move. It's excellent when the background is roughly straight
   under the window, and it needs nothing but the two endpoints.
2. **Full AsLS** (from lesson 3.3) — fit a smooth baseline to the *whole*
   spectrum, subtract it, then integrate. More work, but it handles curved
   backgrounds and many peaks at once.

Let's apply both and grade each against the known true peak area.

In [ ]:
def linear_baseline_area(x, y, lo, hi):
    '''Subtract a straight line between the window endpoints, then integrate.

    Returns (net_area, xm, ym, base) so we can also plot what was removed.
    '''
    m = (x >= lo) & (x <= hi)
    xm, ym = x[m], y[m]
    # Straight line from the first to the last point of the window:
    base = ym[0] + (ym[-1] - ym[0]) * (xm - xm[0]) / (xm[-1] - xm[0])
    net = trapezoid(ym - base, xm)         # area ABOVE the local baseline
    return net, xm, ym, base

lin_area, xm, ym, base = linear_baseline_area(xb, yb, lo, hi)

fig, ax = plt.subplots()
ax.plot(xb, yb, color=C_SIG, lw=1.0, alpha=0.6)
ax.fill_between(xm, ym, base, color=C_FILL, alpha=0.7, label="integrated (net) area")
ax.plot(xm, base, color=C_FALSE, lw=2, label="local linear baseline")
ax.set_xlim(lo-20, hi+20)
ax.set_xlabel(f"{dsb.x_label} ({dsb.x_unit})"); ax.set_ylabel(dsb.y_label)
ax.set_title("Baseline drop: subtract the line, integrate what's left")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "06_baseline_drop.png", dpi=130)
plt.show()

In [ ]:
# AsLS baseline correction, reused verbatim from lesson 3.3.
def asls_baseline(y, lam=1e5, p=0.01, n_iter=10):
    '''Asymmetric Least Squares smoothing baseline (Eilers & Boelens).'''
    L = len(y)
    D = sparse.diags([1, -2, 1], [0, -1, -2], shape=(L, L - 2))
    D = lam * D.dot(D.transpose())          # smoothness penalty
    w = np.ones(L)
    W = sparse.spdiags(w, 0, L, L)
    z = y.copy()
    for _ in range(n_iter):
        W.setdiag(w)
        z = spsolve(W + D, w * y)
        # points above the baseline get little weight; points below get a lot
        w = p * (y > z) + (1 - p) * (y < z)
    return z

# p is the asymmetry: how hard points ABOVE the current baseline are down-weighted.
# A small p (0.001) keeps the fitted baseline from creeping up into a broad band.
asls = asls_baseline(yb, lam=1e6, p=0.001)
yb_corr = yb - asls                          # baseline-corrected spectrum
asls_area = integrate_window(xb, yb_corr, lo, hi)

In [ ]:
# Grade everything against the truth (the true peak area in this same window).
results = pd.DataFrame({
    "method": ["raw (no correction)", "local linear drop", "AsLS (whole spectrum)", "TRUE peak area"],
    "area":   [raw_area, lin_area, asls_area, true_peak_win],
})
results["error_%"] = 100 * (results["area"] - true_peak_win) / true_peak_win
print(results.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

The uncorrected area is badly high; both correction methods land within a percent
or two of the truth. The local linear drop is the cheapest honest answer for an
isolated band on a near-straight background, and AsLS is the more general tool
when the background curves or several peaks share the spectrum. **Either way:
correct the baseline before you integrate.**

## 7. How much do the boundaries matter? (uncertainty)

Your integration limits are never exactly "right" — you, or an automated routine,
pick them with some slack. A good habit is to ask: *if I had drawn the limits a
little differently, how much would the area change?* That spread is a real piece
of measurement uncertainty, and it's easy to estimate by nudging the limits and
watching the answer.

Here's the punchline up front: on a **baseline-corrected** band the answer barely
moves; on the **raw** band it swings, because every nudge adds or removes a slab
of background.

In [ ]:
# Jitter both limits by +/- a few nm around the +/-2 sigma window and collect areas.
rng = np.random.default_rng(0)
jitter_nm = 8.0
n_trials  = 200

raw_areas, corr_areas = [], []
for _ in range(n_trials):
    dlo = rng.uniform(-jitter_nm, jitter_nm)
    dhi = rng.uniform(-jitter_nm, jitter_nm)
    lo_j, hi_j = lo + dlo, hi + dhi
    raw_areas.append(integrate_window(xb, yb, lo_j, hi_j))            # uncorrected
    corr_areas.append(linear_baseline_area(xb, yb, lo_j, hi_j)[0])    # baseline-dropped

raw_areas, corr_areas = np.array(raw_areas), np.array(corr_areas)

def summarize(name, a):
    rsd = 100 * a.std() / a.mean()
    print(f"{name:>22}: mean {a.mean():6.2f}  std {a.std():4.2f}  -> {rsd:4.1f}% spread from limits")

summarize("raw (no correction)", raw_areas)
summarize("local linear drop", corr_areas)

In [ ]:
fig, ax = plt.subplots()
ax.hist(raw_areas,  bins=25, color=C_FALSE, alpha=0.6, label="raw area")
ax.hist(corr_areas, bins=25, color=C_TRUE,  alpha=0.6, label="baseline-corrected area")
ax.axvline(true_peak_win, color="black", lw=2, ls="--", label="true peak area")
ax.set_xlabel("integrated area (AU*nm)"); ax.set_ylabel("count over 200 random limit choices")
ax.set_title("Boundary jitter: baseline correction also stabilises the answer")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "07_boundary_uncertainty.png", dpi=130)
plt.show()

Two payoffs from baseline correction, not one. It removes the *bias* (the raw
histogram sits to the right of the truth) **and** it shrinks the *boundary
sensitivity* (the corrected histogram is far narrower and centred on the truth).
That narrow spread is exactly the number to quote as your integration uncertainty
— and a good reason to record the limits you used.

## 8. Overlapping peaks — one window, two bands

When two bands overlap, a single window around them gives the **combined** area
honestly — area is conserved, so the total is right. What it *cannot* do is tell
you how that total splits between the two. A common field approximation is the
**perpendicular drop**: split at the valley between the peaks and assign each side
to its peak. It's quick, and it's biased whenever the bands genuinely overlap.
Let's see how biased, with the true individual areas as the answer key.

In [ ]:
# Two overlapping bands, clean (so the only error is the splitting method itself).
p1 = (540.0, 40.0, 1.0)
p2 = (585.0, 40.0, 0.7)
dso = uvvis.simulate(peaks=[p1, p2], noise=0.0, baseline=None, seed=2)
xo, yo = dso.x, dso.X[0]

true_a1 = true_gaussian_area(p1[2], p1[1])
true_a2 = true_gaussian_area(p2[2], p2[1])

# Total area over a window spanning both bands -- this part is trustworthy.
lo_o, hi_o = 480.0, 645.0
total_area = integrate_window(xo, yo, lo_o, hi_o)
print(f"combined area (measured) = {total_area:.2f}")
print(f"combined area (true sum) = {true_a1 + true_a2:.2f}   <- area is conserved, total is fine")

In [ ]:
# Perpendicular drop: find the valley (minimum) between the two maxima and split.
win = (xo >= lo_o) & (xo <= hi_o)
xw, yw = xo[win], yo[win]
# the valley is the lowest point strictly between the two peak centres
between = (xw > p1[0]) & (xw < p2[0])
valley_x = xw[between][np.argmin(yw[between])]

left  = xw <= valley_x
a1_drop = trapezoid(yw[left],  xw[left])
a2_drop = trapezoid(yw[~left], xw[~left])

split = pd.DataFrame({
    "band":      ["peak 1 (540)", "peak 2 (585)"],
    "true_area": [true_a1, true_a2],
    "drop_area": [a1_drop, a2_drop],
})
split["error_%"] = 100 * (split["drop_area"] - split["true_area"]) / split["true_area"]
print(split.to_string(index=False, float_format=lambda v: f"{v:.2f}"))

fig, ax = plt.subplots()
ax.plot(xo, yo, color=C_SIG, lw=1.5)
ax.fill_between(xw[left],  yw[left],  color=C_FILL, alpha=0.7)
ax.fill_between(xw[~left], yw[~left], color=C_TRUE, alpha=0.25)
ax.axvline(valley_x, color=C_FALSE, lw=2, ls="--", label=f"perpendicular drop @ {valley_x:.0f} nm")
ax.set_xlabel(f"{dso.x_label} ({dso.x_unit})"); ax.set_ylabel(dso.y_label)
ax.set_title("Perpendicular drop splits the total -- but each side is biased")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "08_overlap_drop.png", dpi=130)
plt.show()

The **total** is essentially exact (area is conserved), but the **split** is
wrong by several percent in each direction: the drop hands part of peak 2's left
tail to peak 1 and vice versa. The taller band steals area from the shorter one.
This is a hard limit of integration — to apportion overlapping bands properly you
have to model their shapes and fit them, the subject of peak fitting (lesson 3.8).
Integrate the *resolved* total when you can; reach for fitting when you can't.

## 9. Why this matters: area, concentration, and calibration

Here is the payoff. Beer-Lambert says band height scales with concentration —
and so, therefore, does band **area**. If we integrate the same band across a
series of known concentrations, area-vs-concentration should be a straight line.
That line *is* a calibration curve: measure an unknown's area, read off its
concentration.

We'll simulate a five-standard series on a sloping baseline (realistic), correct
each baseline, integrate, and check the line against the truth.

In [ ]:
# Five calibration standards at known concentrations. uvvis.simulate scales the
# band height by concentration (Beer-Lambert) but leaves the baseline alone.
concs = np.array([0.2, 0.4, 0.6, 0.8, 1.0])
cal = uvvis.simulate(
    peaks=[band],                      # the (550 nm, FWHM 50, amp 1.0) band
    concentration=concs, n_samples=5,
    baseline={"type": "sloping", "slope": 0.0015, "offset": 0.04},
    noise=0.004, seed=7,
)
xc = cal.x
lo_c, hi_c = center - 1.5*fwhm, center + 1.5*fwhm   # baseline-to-baseline window

# Integrate each standard with a local-linear baseline drop.
areas = np.array([linear_baseline_area(xc, cal.X[i], lo_c, hi_c)[0] for i in range(5)])
# Heights, for comparison (peak max minus the local baseline level).
heights = np.array([cal.X[i][(xc >= lo_c) & (xc <= hi_c)].max() for i in range(5)])

cal_table = pd.DataFrame({"concentration": concs, "area": areas, "height": heights})
print(cal_table.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

In [ ]:
# Fit the calibration line  area = slope * concentration + intercept.
slope, intercept = np.polyfit(concs, areas, 1)
fit = slope * concs + intercept
ss_res = np.sum((areas - fit)**2)
ss_tot = np.sum((areas - areas.mean())**2)
r2 = 1 - ss_res / ss_tot

# The TRUE area-per-unit-concentration is exact: it's the unit-concentration
# band's analytic area.
true_slope = true_gaussian_area(band[2], band[1])

print(f"fitted slope     = {slope:.2f} AU*nm per conc-unit")
print(f"true slope       = {true_slope:.2f}  (exact area of the unit band)")
print(f"intercept        = {intercept:.2f}  (near 0: baseline removed)")
print(f"R^2              = {r2:.5f}")

In [ ]:
fig, ax = plt.subplots()
ax.scatter(concs, areas, color=C_SIG, zorder=3, label="standards (integrated area)")
xs = np.linspace(0, 1.1, 50)
ax.plot(xs, slope*xs + intercept, color=C_TRUE, label=f"fit: area = {slope:.1f} c + {intercept:.2f}  (R^2={r2:.4f})")
ax.set_xlabel("concentration (arb. units)"); ax.set_ylabel("integrated area (AU*nm)")
ax.set_title("Calibration curve: integrated area is linear in concentration")
ax.legend()
fig.tight_layout(); fig.savefig(EXPORTS / "09_calibration.png", dpi=130)
plt.show()

In [ ]:
# Use the calibration to quantify an UNKNOWN of (secretly) known concentration.
true_unknown_conc = 0.73
unk = uvvis.simulate(
    peaks=[band], concentration=true_unknown_conc, n_samples=1,
    baseline={"type": "sloping", "slope": 0.0015, "offset": 0.04},
    noise=0.004, seed=99,
)
unk_area = linear_baseline_area(unk.x, unk.X[0], lo_c, hi_c)[0]

# Invert the line: concentration = (area - intercept) / slope.
pred_conc = (unk_area - intercept) / slope
print(f"unknown integrated area = {unk_area:.2f} AU*nm")
print(f"predicted concentration = {pred_conc:.3f}")
print(f"true concentration      = {true_unknown_conc:.3f}")
print(f"error                   = {100*(pred_conc-true_unknown_conc)/true_unknown_conc:+.1f}%")

In [ ]:
# Why area over height? When band SHAPE drifts run-to-run (temperature, optics),
# height moves even at fixed amount -- area doesn't. Simulate five runs of the
# SAME true amount whose width varies, and compare.
fixed_area = 45.0
widths = np.array([35, 42, 50, 58, 66.0])
print(f"{'FWHM(nm)':>9} {'height':>8} {'area':>8}")
for w in widths:
    amp = fixed_area / (w * AREA_K)            # amplitude chosen to hold area fixed
    yw = gaussian(x, 550.0, w, amp) + np.random.default_rng(int(w)).normal(0, 0.003, x.size)
    h = yw.max()
    # Integrate each band over ITS OWN baseline-to-baseline window (+/-1.5*FWHM),
    # so a broad band isn't truncated -- a fixed narrow window would fake a drop.
    a = linear_baseline_area(x, yw, 550.0 - 1.5*w, 550.0 + 1.5*w)[0]
    print(f"{w:>9.0f} {h:>8.3f} {a:>8.2f}")
print("\nSame amount every row. Height varies ~2x; area stays put. That is why")
print("quantitative methods report PEAK AREA.")

The calibration is essentially perfect (`R^2 > 0.999`), the fitted slope matches
the exact area of the unit band to within ~1 %, and the unknown is recovered
within a percent or two. And the last table closes the loop back to section 2: when the band shape
wanders but the amount is fixed, **area holds steady while height does not.** That
is the case for integration in one line.

## 10. A reusable `integrate_peak()` helper

Let's bottle the method we converged on — local-linear baseline subtraction plus
the trapezoidal rule — into one small function the later notebooks can import.
Sensible defaults, baseline handling built in, and an option to return the
diagnostics you'd want to plot or report.

In [ ]:
def integrate_peak(x, y, lo, hi, baseline="linear", return_details=False):
    '''Integrate a band over [lo, hi], with the baseline handled honestly.

    Parameters
    ----------
    x, y : arrays
        The spectrum (wavelength axis and signal).
    lo, hi : float
        Integration limits, in axis units (e.g. nm).
    baseline : {"linear", None}
        "linear" subtracts a straight line between the window endpoints (the
        baseline-drop method) before integrating; None integrates the raw signal
        (use only if the spectrum is already baseline-corrected).
    return_details : bool
        If True, also return a dict with the local x/y, the baseline used, and
        the limits -- handy for plotting or a method report.

    Returns
    -------
    area : float
        The net integrated area (above the baseline).
    details : dict, optional
        Only if return_details=True.
    '''
    m = (x >= lo) & (x <= hi)             # select the integration window
    xm, ym = x[m], y[m]

    if baseline == "linear":
        # straight line between the first and last point of the window
        base = ym[0] + (ym[-1] - ym[0]) * (xm - xm[0]) / (xm[-1] - xm[0])
    elif baseline is None:
        base = np.zeros_like(ym)          # assume already baseline-corrected
    else:
        raise ValueError("baseline must be 'linear' or None")

    area = trapezoid(ym - base, xm)       # trapezoidal area above the baseline

    if return_details:
        return area, {"x": xm, "y": ym, "baseline": base, "lo": lo, "hi": hi}
    return area

# Quick self-check against the true area we already know.
check = integrate_peak(xb, yb, lo, hi, baseline="linear")
print(f"integrate_peak() area = {check:.2f}   (true peak area {true_peak_win:.2f})")
print(f"agreement             = {100*check/true_peak_win:.1f}% of truth")

That's a function you can trust on an isolated, baseline-correctable band — and a
clear signpost for when you *can't* (heavy overlap), where fitting takes over.

## Key Takeaways

- **Area answers "how much"; height answers "how tall."** They agree only while
  the band shape is fixed. When shape drifts — broadening, saturation — area stays
  proportional to amount and height does not. That's why quantitation uses area.
- **Numerical integration is the easy part.** The trapezoidal rule (`np.trapezoid`)
  is exact enough once the band is adequately sampled. Error comes from your
  *limits* and your *baseline*, not the rule.
- **Limits are a decision.** Too narrow truncates the tails; too wide scoops up
  neighbours and background. A `+/-2 sigma` window captures ~95 % of an isolated
  band — but when you'll subtract a baseline, run the limits out to where the band
  meets the background (~`+/-1.5 FWHM`, ≈ `+/-3.5 sigma`) so the endpoints anchor
  on baseline, not on the peak's tail.
- **An uncorrected baseline biases every area — upward, and worse for wider
  windows.** Integration sums the background right along with the peak.
- **Correct the baseline before integrating.** A local linear "drop" between the
  window endpoints is the cheap, honest fix for a near-straight background; AsLS
  (3.3) handles curved backgrounds and many peaks at once.
- **Report boundary uncertainty.** Nudging the limits and watching the area spread
  gives a real error bar; baseline correction shrinks that spread as well as the
  bias.
- **Overlapping bands: the total is conserved, the split is not.** Perpendicular
  drop apportions a combined area only approximately; true separation needs
  fitting (3.8).
- **Area is linear in concentration.** Integrate known standards, fit the line,
  invert it for unknowns — that's quantitation.

## Practical Checklist

- [ ] **Baseline-correct first** (local linear drop or AsLS), *then* integrate.
- [ ] **Set limits deliberately** — from the band's FWHM, not by eye; run them to
  where the band meets the baseline (~`+/-1.5 FWHM`) so endpoints sit on background.
- [ ] **Keep the limits consistent** across every sample in a series.
- [ ] **Sample the band well** — a handful of points across it undercounts the area.
- [ ] **Check sensitivity** — jitter the limits a little; if the area swings, your
  baseline or window needs work.
- [ ] **For overlaps, integrate the resolved total**; only fit (3.8) when you must
  split.
- [ ] **Report area in axis units** (e.g. AU*nm) with the limits and baseline
  method.

## Common Mistakes

- **Calling `np.trapezoid(y, x)` on the raw spectrum** and trusting the number —
  you've integrated the background too.
- **Widening the window to "be safe."** With an uncorrected baseline that makes
  the bias *larger*, not smaller.
- **Using height for a band whose shape changes** run-to-run, then wondering why
  the calibration drifts.
- **Different limits for different samples** in the same series — areas are no
  longer comparable.
- **Splitting overlapping peaks by perpendicular drop and reporting the pieces as
  exact.** The total is fine; the split is biased.
- **Not recording the integration limits and baseline method**, so nobody
  (including future-you) can reproduce the area.

## Reporting Guidance

State exactly how the area was produced: **"Peak area integrated over [lo-hi nm]
by the trapezoidal rule after a local linear baseline subtraction between the
window endpoints"** (or **"after AsLS baseline correction, lambda = ..., p =
..."**). Give the area in its units (e.g. AU*nm), the limits, and — where it
matters — the boundary-sensitivity spread as an uncertainty. For quantitation,
report the calibration: slope, intercept, R^2, the concentration range covered,
and the integration settings, since those settings define what "area" means and
must be reproducible.

## Next Lesson

**3.6 — Signal Averaging and the √N Rule: When More Scans Help and When They
Don't.** You can now turn a peak into an honest number; next we step back to
acquisition — how co-adding scans improves signal-to-noise by √N, and when
drift or a changing sample makes more scans stop helping. (Peak fitting, which
pulls overlapping bands apart, is the advanced lesson 3.8.)